# CS-781 Capstone Project — Phase 7: Prior Case Retrieval (PCR)
## Champion High-Performance Multi-Method Pipeline (Targeting Micro-F1@7 >= 0.55–0.70+)
### Pure PyTorch Dual-Encoder Architecture with Hybrid Neural-Lexical Representation & Reciprocal Rank Fusion

---

### Project Governance & Course Rules Compliance:
1. **Strict Prohibition on Hugging Face Model Classes:**
   - Hugging Face model classes (`AutoModel`, `BertModel`, `DPRQuestionEncoder`, `SentenceTransformer`, etc.) are **STRICTLY PROHIBITED**.
   - All backbones, PEFT adapters, attention layers, and projection heads are implemented using **pure PyTorch primitives** (`torch.nn.Module`, `torch.nn.MultiheadAttention`, `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.LayerNorm`).
   - Hugging Face is used **ONLY** for tokenization (`AutoTokenizer.from_pretrained("openai-community/gpt2")`).
2. **Phase 1 Foundation Model Starting Point:**
   - Every experiment builds upon the Phase 1 Legal Language Model backbone (`champion_moe.pth` / `moe_backup_epoch4_COMPLETE.pth`).
   - Adapts the 6-layer Hybrid Mixture-of-Experts (MoE) model into a bidirectional dual-encoder representation model via attention-masked mean pooling and representation projection.
3. **Mandatory 6-Experiment Rule (Rubric Section 13.2):**
   - **Method 1 (`bitfit` - P7-1):** Bias-only fine-tuning (~0.1% trainable parameters).
   - **Method 2 (`layer_freezing` - P7-2):** Freezes lower 4 layers, fine-tunes top 2 blocks + projection head.
   - **Method 3 (`bottleneck_adapter` - P7-3):** Series bottleneck adapters ($768 \to 64 \to 768$ with GELU) in attention and FFN blocks.
   - **Method 4 (`adalora` - P7-4):** Adaptive Low-Rank SVD parameterization with dynamic rank allocation.
   - **Method 5 (`lorma` - P7-5):** **COMPULSORY PEFT CHAMPION:** Low-Rank Multiplicative Adaptation ($h = W_0 x + \alpha \cdot B(A(W_0 x))$).
   - **Method 6 (`distill` - P7-6):** **COMPULSORY KNOWLEDGE DISTILLATION:** Distills from the fine-tuned LoRMA dual-encoder into a compact 3-layer student model (`CompactLegalStudentForPCR`).
   - **Bonus Method (`ensemble`):** Reciprocal Rank Fusion (RRF) across top models -> `submission_(ensemble).zip` and root `submission.zip`.
4. **Strict Schema & Top-7 Invariants (Section 13.3.5 & 13.4):**
   - `prediction.json` at root of `submission.zip`.
   - JSON dictionary mapping each query ID to **EXACTLY 7 UNIQUE** candidate case IDs:
     `{"query_id": ["cand_1", "cand_2", "cand_3", "cand_4", "cand_5", "cand_6", "cand_7"]}`
   - Candidate IDs must belong strictly to the corresponding `test_candidates.json` pool.
   - Ordered from highest to lowest predicted relevance.
5. **Automated One-Click Execution:**
   - Click **"Run All"**: Automatically executes all 6 methods back-to-back, manages GPU memory, saves every model checkpoint, and exports all required submission files in `submission_(methodname).zip` format!


In [ ]:
# Step 0: Environment Setup & Deterministic Seeds
import os
import sys
import gc
import json
import math
import time
import random
import re
import zipfile
import shutil
import glob
from collections import Counter
from typing import Dict, List, Any, Tuple, Optional, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"PyTorch Version: {torch.__version__}")
print(f"Primary Device : {device}")
if torch.cuda.is_available():
    print(f"GPU Name       : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 70)


In [ ]:
# Step 1: Pretrained Phase 1 Backbone Checkpoint Discovery
SAVED_CKPT_PATH = "/kaggle/working/champion_moe.pth"

def prepare_checkpoint():
    # 1. Check if already in working directory
    if os.path.exists(SAVED_CKPT_PATH) and os.path.getsize(SAVED_CKPT_PATH) > 100 * 1024 * 1024:
        return SAVED_CKPT_PATH

    # 2. Check for uncompressed directory containing data.pkl
    data_pkl_matches = glob.glob("/kaggle/input/**/data.pkl", recursive=True)
    if data_pkl_matches:
        ckpt_dir = os.path.dirname(data_pkl_matches[0])
        print(f"[Safeguard] Detected unpacked checkpoint directory: {ckpt_dir}")
        print(f"[Safeguard] Packaging into PyTorch format at {SAVED_CKPT_PATH}...")
        with zipfile.ZipFile(SAVED_CKPT_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
            for root, dirs, files in os.walk(ckpt_dir):
                for f in files:
                    full_p = os.path.join(root, f)
                    rel_p = os.path.relpath(full_p, ckpt_dir).replace(os.sep, "/")
                    arc_p = rel_p if rel_p.startswith("archive/") else f"archive/{rel_p}"
                    zf.write(full_p, arcname=arc_p)
        return SAVED_CKPT_PATH

    # 3. Check /kaggle/input for any .pth file or checkpoint
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            full_p = os.path.join(root, f)
            if not f.endswith((".json", ".csv", ".txt", ".py", ".md", ".png")):
                if os.path.isfile(full_p) and os.path.getsize(full_p) > 100 * 1024 * 1024:
                    print(f"[Safeguard] Found model checkpoint at {full_p}. Copying to {SAVED_CKPT_PATH}...")
                    shutil.copy(full_p, SAVED_CKPT_PATH)
                    return SAVED_CKPT_PATH

    return None

ckpt_src = prepare_checkpoint()
if ckpt_src:
    size_mb = os.path.getsize(ckpt_src) / (1024 * 1024)
    print("=" * 65)
    print(f"[SUCCESS] PHASE 1 BACKBONE READY: {ckpt_src} ({size_mb:.1f} MB)")
    print("=" * 65)
else:
    print("=" * 65)
    print("[WARNING] Checkpoint not found in /kaggle/input yet! Will use clean initialization.")
    print("=" * 65)


In [ ]:
# Step 2: Acquire Official Phase 7 PCR Dataset (Hugging Face Hub or Kaggle Input)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

hf_token = os.environ.get("HF_TOKEN", None)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Detected HF_TOKEN from Kaggle Secrets.")
except Exception:
    pass

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF")
        print("Detected HF token from Kaggle Secrets.")
    except Exception:
        pass

if hf_token:
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("Logged into Hugging Face successfully!")
    except Exception as e:
        print(f"HF login note: {e}")

files_to_download = [
    "Phase_7_PCR/Train/train.json",
    "Phase_7_PCR/Train/train_candidates.json",
    "Phase_7_PCR/Dev/dev.json",
    "Phase_7_PCR/Dev/dev_candidates.json",
    "Phase_7_PCR/Test/test.json",
    "Phase_7_PCR/Test/test_candidates.json"
]

data_paths = {}
for rel_path in files_to_download:
    fname = os.path.basename(rel_path)
    # Check if a file with similar name is in /kaggle/input/
    local_matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if local_matches:
        data_paths[rel_path] = local_matches[0]
        print(f"[Dataset] Found local file in /kaggle/input/: {local_matches[0]}")
        continue
    if os.path.exists(fname):
        data_paths[rel_path] = fname
        print(f"[Dataset] Found in working directory: {fname}")
        continue
    try:
        from huggingface_hub import hf_hub_download
        local_path = hf_hub_download(repo_id=HF_REPO_ID, filename=rel_path, repo_type="dataset", token=hf_token)
        data_paths[rel_path] = local_path
        print(f"[Dataset] Downloaded from HF: {rel_path} -> {local_path}")
    except Exception as e:
        print(f"[Dataset] Download notice for {rel_path}: {e}")


In [ ]:
# Step 3: Parse Dataset & Align Query-Candidate Pools
def extract_text(raw_val) -> str:
    """Extracts clean string text from string, list of sentences, or dict."""
    if isinstance(raw_val, str):
        return raw_val.strip()
    elif isinstance(raw_val, list):
        return " ".join([str(s) for s in raw_val if s]).strip()
    elif isinstance(raw_val, dict):
        t = raw_val.get("text", raw_val.get("document", raw_val.get("case_text", "")))
        return extract_text(t)
    return str(raw_val).strip()

def parse_pcr_split(query_path, cand_path):
    """Parses query file and corresponding candidate pool for a split."""
    queries = {}
    candidates = {}

    if cand_path and os.path.exists(cand_path):
        with open(cand_path, "r", encoding="utf-8") as f:
            cand_raw = json.load(f)
        if isinstance(cand_raw, dict):
            for cid, cdoc in cand_raw.items():
                candidates[str(cid)] = extract_text(cdoc)
        elif isinstance(cand_raw, list):
            for item in cand_raw:
                if isinstance(item, dict):
                    cid = str(item.get("id", item.get("case_id", len(candidates))))
                    candidates[cid] = extract_text(item)

    if query_path and os.path.exists(query_path):
        with open(query_path, "r", encoding="utf-8") as f:
            q_raw = json.load(f)
        if isinstance(q_raw, dict):
            for qid, qdata in q_raw.items():
                if isinstance(qdata, dict):
                    qtext = extract_text(qdata.get("text", qdata.get("document", "")))
                    rel = [str(x) for x in qdata.get("relevant_candidates", qdata.get("relevant", []))]
                else:
                    qtext = extract_text(qdata)
                    rel = []
                queries[str(qid)] = {"text": qtext, "relevant": rel}
        elif isinstance(q_raw, list):
            for item in q_raw:
                if isinstance(item, dict):
                    qid = str(item.get("id", item.get("case_id", len(queries))))
                    qtext = extract_text(item.get("text", item.get("document", "")))
                    rel = [str(x) for x in item.get("relevant_candidates", item.get("relevant", []))]
                    queries[qid] = {"text": qtext, "relevant": rel}

    return queries, candidates

train_queries, train_candidates = parse_pcr_split(
    data_paths.get("Phase_7_PCR/Train/train.json", ""),
    data_paths.get("Phase_7_PCR/Train/train_candidates.json", "")
)
dev_queries, dev_candidates = parse_pcr_split(
    data_paths.get("Phase_7_PCR/Dev/dev.json", ""),
    data_paths.get("Phase_7_PCR/Dev/dev_candidates.json", "")
)
test_queries, test_candidates = parse_pcr_split(
    data_paths.get("Phase_7_PCR/Test/test.json", ""),
    data_paths.get("Phase_7_PCR/Test/test_candidates.json", "")
)

# Build Training Pairs: (query_text, positive_candidate_text)
train_pairs = []
for qid, qinfo in train_queries.items():
    qtext = qinfo["text"]
    for rel_cid in qinfo["relevant"]:
        if rel_cid in train_candidates:
            train_pairs.append({
                "qid": qid,
                "cid": rel_cid,
                "qtext": qtext,
                "ctext": train_candidates[rel_cid]
            })

# Limit training pairs for high throughput (trains in ~3-4 mins per method)
MAX_TRAIN_PAIRS = 2000
if len(train_pairs) > MAX_TRAIN_PAIRS:
    random.seed(42)
    train_pairs = random.sample(train_pairs, MAX_TRAIN_PAIRS)

print("=" * 70)
print("PHASE 7 (PRIOR CASE RETRIEVAL) DATASET INVENTORY:")
print(f"  Training Queries     : {len(train_queries):,} | Train Candidate Pool: {len(train_candidates):,}")
print(f"  Active Train Pairs   : {len(train_pairs):,} (Query + Positive Prior Case)")
print(f"  Dev Queries          : {len(dev_queries):,} | Dev Candidate Pool  : {len(dev_candidates):,}")
print(f"  Test Queries         : {len(test_queries):,} | Test Candidate Pool : {len(test_candidates):,}")
print("=" * 70)


In [ ]:
# Step 4: Fast Lexical Prior (TF-IDF / BM25 Legal Citation Overlap)
print("Building Lexical Retrieval Indices for Dev and Test pools...")
t0_lex = time.time()

try:
    from sklearn.feature_extraction.text import TfidfVectorizer

    def build_lexical_similarity_matrix(query_texts, cand_texts, max_features=12000):
        """Computes cosine similarity matrix over legal sublinear TF-IDF."""
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            stop_words="english",
            sublinear_tf=True
        )
        cand_mat = vectorizer.fit_transform(cand_texts)
        query_mat = vectorizer.transform(query_texts)
        # Cosine similarity matrix: (N_query, N_cand)
        sim_mat = (query_mat @ cand_mat.T).toarray()
        return sim_mat

    # Pre-index Dev lexical similarities
    dev_q_ids = sorted(list(dev_queries.keys()))
    dev_c_ids = sorted(list(dev_candidates.keys()))
    dev_q_texts = [dev_queries[qid]["text"] for qid in dev_q_ids]
    dev_c_texts = [dev_candidates[cid] for cid in dev_c_ids]
    dev_lex_sim = build_lexical_similarity_matrix(dev_q_texts, dev_c_texts) if dev_q_ids and dev_c_ids else None

    # Pre-index Test lexical similarities
    test_q_ids = sorted(list(test_queries.keys()))
    test_c_ids = sorted(list(test_candidates.keys()))
    test_q_texts = [test_queries[qid]["text"] for qid in test_q_ids]
    test_c_texts = [test_candidates[cid] for cid in test_c_ids]
    test_lex_sim = build_lexical_similarity_matrix(test_q_texts, test_c_texts) if test_q_ids and test_c_ids else None

    has_lexical = True
    print(f"[SUCCESS] Lexical Similarity Matrices computed in {time.time()-t0_lex:.1f}s!")
except Exception as e:
    print(f"[Lexical Prior Note] {e} -> Continuing with pure dense dual-encoder.")
    has_lexical = False
    dev_lex_sim = None
    test_lex_sim = None


In [ ]:
# Step 5: Long-Document Salient Section Windowing & DataLoaders
MAX_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 100000

def extract_salient_legal_case(doc_text: str, max_words=320) -> str:
    """
    Extracts high-salience legal context:
    - Lead 200 words (charges, parties, dispute, jurisdiction)
    - Tail 120 words (operative decree, conviction, precedents relied upon)
    """
    words = doc_text.split()
    if len(words) <= max_words:
        return " ".join(words)
    lead_w = words[:200]
    tail_w = words[-120:]
    return " ".join(lead_w) + " ... [Holding]: " + " ".join(tail_w)

class PCRPairDataset(Dataset):
    """Dataset for training dual-encoders with query-candidate positive pairs."""
    def __init__(self, pairs_list, tokenizer, max_len=MAX_TOKENS):
        self.samples = []
        pad_id = tokenizer.pad_token_id or 0

        for item in pairs_list:
            q_text = extract_salient_legal_case(item["qtext"])
            c_text = extract_salient_legal_case(item["ctext"])

            q_toks = tokenizer.encode(q_text, add_special_tokens=True, truncation=True, max_length=max_len)
            c_toks = tokenizer.encode(c_text, add_special_tokens=True, truncation=True, max_length=max_len)

            q_pad = max_len - len(q_toks)
            c_pad = max_len - len(c_toks)

            q_ids = q_toks + [pad_id] * q_pad
            q_mask = [1] * len(q_toks) + [0] * q_pad

            c_ids = c_toks + [pad_id] * c_pad
            c_mask = [1] * len(c_toks) + [0] * c_pad

            self.samples.append({
                "q_input_ids": torch.tensor(q_ids, dtype=torch.long),
                "q_attention_mask": torch.tensor(q_mask, dtype=torch.long),
                "c_input_ids": torch.tensor(c_ids, dtype=torch.long),
                "c_attention_mask": torch.tensor(c_mask, dtype=torch.long)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def pcr_collate_fn(batch):
    return {
        "q_input_ids": torch.stack([x["q_input_ids"] for x in batch]),
        "q_attention_mask": torch.stack([x["q_attention_mask"] for x in batch]),
        "c_input_ids": torch.stack([x["c_input_ids"] for x in batch]),
        "c_attention_mask": torch.stack([x["c_attention_mask"] for x in batch])
    }

print("Tokenizing PCR training pairs...")
train_dataset = PCRPairDataset(train_pairs, tokenizer, max_len=MAX_TOKENS)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=pcr_collate_fn,
    pin_memory=True,
    drop_last=True
)
print(f"DataLoaders Ready! Batches per epoch: {len(train_loader):,}")


In [ ]:
# Step 6: Pure PyTorch Dual-Encoder Architecture & PEFT Suite
class StandardExpert(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim)
        )
    def forward(self, x): return self.net(x)

class MoERouter(nn.Module):
    def __init__(self, dim, num_experts=4):
        super().__init__()
        self.gate = nn.Linear(dim, num_experts, bias=False)
        self.num_experts = num_experts

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        weights, indices = torch.max(probs, dim=-1)
        return weights, indices

class MoELayer(nn.Module):
    def __init__(self, dim, num_experts=4):
        super().__init__()
        self.num_experts = num_experts
        self.router = MoERouter(dim, num_experts)
        self.experts = nn.ModuleList([StandardExpert(dim) for _ in range(num_experts)])

    def forward(self, x):
        b, seq, d = x.shape
        flat_x = x.view(-1, d)
        weights, indices = self.router(flat_x)
        out = torch.zeros_like(flat_x)
        for i, expert in enumerate(self.experts):
            mask = (indices == i)
            if mask.any():
                exp_out = expert(flat_x[mask])
                out[mask] = exp_out * weights[mask].unsqueeze(-1)
        return out.view(b, seq, d)

class HybridBlock(nn.Module):
    def __init__(self, dim, heads, is_moe=False):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = MoELayer(dim, num_experts=4) if is_moe else StandardExpert(dim)
        self.attn_adapter = None
        self.ffn_adapter = None

    def forward(self, x, mask=None):
        n_x = self.norm1(x)
        attn_out, _ = self.attn(n_x, n_x, n_x, attn_mask=mask)
        if hasattr(self, "attn_adapter") and self.attn_adapter is not None:
            attn_out = self.attn_adapter(attn_out)
        x = x + attn_out

        ffn_out = self.ffn(self.norm2(x))
        if hasattr(self, "ffn_adapter") and self.ffn_adapter is not None:
            ffn_out = self.ffn_adapter(ffn_out)
        x = x + ffn_out
        return x

class HybridMoE(nn.Module):
    def __init__(self, vocab_count=50257, embed_size=768, heads=12, layers=6):
        super().__init__()
        self.embed_size = embed_size
        self.tok_emb = nn.Embedding(vocab_count, embed_size)
        self.pos_emb = nn.Embedding(512, embed_size)
        self.blocks = nn.ModuleList([
            HybridBlock(embed_size, heads, is_moe=(i % 2 != 0)) for i in range(layers)
        ])
        self.final_norm = nn.LayerNorm(embed_size)
        self.output_head = nn.Linear(embed_size, vocab_count, bias=False)
        self.tok_emb.weight = self.output_head.weight

    def get_hidden_states(self, x, causal=False):
        seq_len = x.shape[1]
        pos = torch.arange(seq_len, device=x.device)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float("-inf"), diagonal=1) if causal else None
        out = self.tok_emb(x) + self.pos_emb(pos)
        for block in self.blocks:
            out = block(out, mask)
        return self.final_norm(out)

class HybridMoEForPCR(nn.Module):
    """
    Dual-Encoder for Prior Case Retrieval.
    Transforms Phase 1 backbone into a bidirectional semantic retriever with projection head.
    """
    def __init__(self, backbone, proj_dim=256):
        super().__init__()
        self.backbone = backbone
        self.embed_size = backbone.embed_size
        self.proj = nn.Sequential(
            nn.Linear(self.embed_size, proj_dim),
            nn.LayerNorm(proj_dim)
        )

    def forward(self, input_ids, attention_mask):
        h = self.backbone.get_hidden_states(input_ids, causal=False)
        # Attention-masked mean pooling
        mask_exp = attention_mask.unsqueeze(-1).expand_as(h).float()
        sum_h = torch.sum(h * mask_exp, dim=1)
        sum_m = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        doc_emb = sum_h / sum_m
        # Project and L2-normalize
        p_emb = self.proj(doc_emb)
        return F.normalize(p_emb, p=2, dim=-1)

def load_phase1_backbone(checkpoint_path, device="cpu"):
    backbone = HybridMoE(vocab_count=50257, embed_size=768, heads=12, layers=6)
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            state_dict = torch.load(checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            state_dict = torch.load(checkpoint_path, map_location=device)
        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        elif isinstance(state_dict, dict) and "model" in state_dict:
            state_dict = state_dict["model"]
        clean_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items() if not k.startswith("output_head") or k == "output_head.weight"}
        backbone.load_state_dict(clean_state_dict, strict=False)
        print(f"[Backbone] Successfully loaded Phase 1 backbone weights from: {checkpoint_path}")
    else:
        print("[Backbone Note] Checkpoint not found, using clean initialization.")
    return backbone

# ----------------------------------------------------------------------------
# PEFT Modules
# ----------------------------------------------------------------------------
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=1.0, dropout=0.05):
        super().__init__()
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
        self.r = r
        self.alpha = alpha
        self.lorma_A = nn.Linear(original_linear.out_features, r, bias=False)
        self.lorma_B = nn.Linear(r, original_linear.out_features, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        nn.init.normal_(self.lorma_A.weight, std=1.0 / math.sqrt(r))
        nn.init.zeros_(self.lorma_B.weight)

    @property
    def weight(self): return self.original_linear.weight
    @property
    def bias(self): return self.original_linear.bias

    def forward(self, x):
        y0 = self.original_linear(x)
        return y0 + self.lorma_B(self.lorma_A(self.dropout(y0))) * self.alpha

class AdaLoRALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32.0, dropout=0.05):
        super().__init__()
        self.original_linear = original_linear
        self.original_linear.weight.requires_grad = False
        if self.original_linear.bias is not None:
            self.original_linear.bias.requires_grad = False
        self.r = r
        self.scaling = alpha / r
        in_dim = original_linear.in_features
        out_dim = original_linear.out_features
        self.lora_P = nn.Linear(in_dim, r, bias=False)
        self.lora_E = nn.Parameter(torch.ones(r))
        self.lora_Q = nn.Linear(r, out_dim, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        nn.init.kaiming_uniform_(self.lora_P.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_Q.weight)

    @property
    def weight(self): return self.original_linear.weight
    @property
    def bias(self): return self.original_linear.bias

    def forward(self, x):
        base_out = self.original_linear(x)
        lora_out = self.lora_Q(self.lora_P(self.dropout(x)) * self.lora_E)
        return base_out + lora_out * self.scaling

class BottleneckAdapter(nn.Module):
    def __init__(self, dim, bottleneck_dim=64):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, bottleneck_dim),
            nn.GELU(),
            nn.Linear(bottleneck_dim, dim)
        )
        nn.init.normal_(self.adapter[1].weight, std=1e-2)
        nn.init.zeros_(self.adapter[3].weight)
        nn.init.zeros_(self.adapter[3].bias)

    def forward(self, x): return x + self.adapter(x)

class CompactLegalStudentForPCR(nn.Module):
    """Compact 3-layer Student Model for PCR Knowledge Distillation."""
    def __init__(self, vocab_count=50257, embed_size=384, heads=6, layers=3, proj_dim=256):
        super().__init__()
        self.embed_size = embed_size
        self.tok_emb = nn.Embedding(vocab_count, embed_size)
        self.pos_emb = nn.Embedding(512, embed_size)
        self.blocks = nn.ModuleList([
            HybridBlock(embed_size, heads, is_moe=False) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(embed_size)
        self.proj = nn.Sequential(
            nn.Linear(embed_size, proj_dim),
            nn.LayerNorm(proj_dim)
        )

    def forward(self, input_ids, attention_mask):
        seq_len = input_ids.shape[1]
        pos = torch.arange(seq_len, device=input_ids.device)
        out = self.tok_emb(input_ids) + self.pos_emb(pos)
        for block in self.blocks:
            out = block(out, mask=None)
        h = self.final_norm(out)
        mask_exp = attention_mask.unsqueeze(-1).expand_as(h).float()
        sum_h = torch.sum(h * mask_exp, dim=1)
        sum_m = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        doc_emb = sum_h / sum_m
        return F.normalize(self.proj(doc_emb), p=2, dim=-1)

def configure_pcr_adaptation(model, method="lorma", r=16):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.proj.parameters():
        p.requires_grad = True

    if method == "bitfit":
        for name, param in model.named_parameters():
            if "bias" in name or "norm" in name:
                param.requires_grad = True

    elif method == "layer_freezing":
        for name, param in model.named_parameters():
            if "blocks.4." in name or "blocks.5." in name or "final_norm" in name:
                param.requires_grad = True

    elif method == "bottleneck_adapter":
        for block in model.backbone.blocks:
            block.attn_adapter = BottleneckAdapter(model.embed_size, bottleneck_dim=64)
            block.ffn_adapter = BottleneckAdapter(model.embed_size, bottleneck_dim=64)
            for p in block.attn_adapter.parameters(): p.requires_grad = True
            for p in block.ffn_adapter.parameters(): p.requires_grad = True
        for p in model.backbone.final_norm.parameters(): p.requires_grad = True

    elif method == "adalora":
        for block in model.backbone.blocks:
            for p in block.norm1.parameters(): p.requires_grad = True
            for p in block.norm2.parameters(): p.requires_grad = True
            if isinstance(block.ffn, StandardExpert):
                block.ffn.net[0] = AdaLoRALinear(block.ffn.net[0], r=r, alpha=32.0, dropout=0.05)
                block.ffn.net[2] = AdaLoRALinear(block.ffn.net[2], r=r, alpha=32.0, dropout=0.05)
        for p in model.backbone.final_norm.parameters(): p.requires_grad = True

    elif method == "lorma":
        for block in model.backbone.blocks:
            for p in block.norm1.parameters(): p.requires_grad = True
            for p in block.norm2.parameters(): p.requires_grad = True
            if isinstance(block.ffn, StandardExpert):
                block.ffn.net[0] = LoRMALinear(block.ffn.net[0], r=r, alpha=1.0, dropout=0.05)
                block.ffn.net[2] = LoRMALinear(block.ffn.net[2], r=r, alpha=1.0, dropout=0.05)
            elif isinstance(block.ffn, MoELayer):
                for expert in block.ffn.experts:
                    expert.net[0] = LoRMALinear(expert.net[0], r=r, alpha=1.0, dropout=0.05)
                    expert.net[2] = LoRMALinear(expert.net[2], r=r, alpha=1.0, dropout=0.05)
        for p in model.backbone.final_norm.parameters(): p.requires_grad = True

    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  [PEFT Config] Method: {method.upper()} | Trainable: {trainable_p:,}/{total_p:,} ({100*trainable_p/total_p:.2f}%)")
    return model


In [ ]:
# Step 7: InfoNCE Contrastive Loss & Official Micro-F1@7 Evaluator
class InfoNCEContrastiveLoss(nn.Module):
    """Symmetric In-Batch Contrastive Loss with temperature scaling."""
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature

    def forward(self, q_emb, c_emb):
        # Similarity matrix: (B, B)
        sim = torch.matmul(q_emb, c_emb.T) / self.temperature
        targets = torch.arange(q_emb.size(0), device=q_emb.device)
        loss_q = F.cross_entropy(sim, targets)
        loss_c = F.cross_entropy(sim.T, targets)
        return 0.5 * (loss_q + loss_c)

def compute_micro_f1_at_7(gold_map: Dict[str, List[str]], pred_map: Dict[str, List[str]]) -> Tuple[float, float, float]:
    """Computes exact Micro-F1@7, Micro-Precision@7, and Micro-Recall@7."""
    total_tp = 0
    total_pred = 0
    total_gold = 0

    for qid, preds in pred_map.items():
        top7_preds = preds[:7]
        gold_set = set(gold_map.get(qid, []))
        tp = sum(1 for p in top7_preds if p in gold_set)

        total_tp += tp
        total_pred += len(top7_preds)
        total_gold += len(gold_set)

    micro_prec = total_tp / max(1, total_pred)
    micro_rec = total_tp / max(1, total_gold)
    micro_f1 = (2 * micro_prec * micro_rec) / max(1e-9, micro_prec + micro_rec)
    return micro_f1, micro_prec, micro_rec

def encode_texts_into_matrix(model, texts_list, tokenizer, batch_size=32, device="cuda") -> np.ndarray:
    """Encodes a list of legal case texts into normalized dense embeddings."""
    model.eval()
    embeddings = []
    pad_id = tokenizer.pad_token_id or 0

    for start in range(0, len(texts_list), batch_size):
        batch_texts = texts_list[start : start + batch_size]
        encodings = [tokenizer.encode(extract_salient_legal_case(t), add_special_tokens=True, truncation=True, max_length=MAX_TOKENS) for t in batch_texts]

        max_l = max(len(e) for e in encodings)
        ids = np.full((len(batch_texts), max_l), pad_id, dtype=np.int64)
        masks = np.zeros((len(batch_texts), max_l), dtype=np.int64)

        for i, e in enumerate(encodings):
            ids[i, :len(e)] = e
            masks[i, :len(e)] = 1

        with torch.no_grad():
            with autocast_ctx():
                emb = model(torch.tensor(ids, device=device), torch.tensor(masks, device=device))
            embeddings.append(emb.cpu().numpy())

    return np.vstack(embeddings) if embeddings else np.empty((0, 256))


In [ ]:
# Step 8: Automated High-Precision Master Runner (Trains All Required Methods)
RUN_ALL_METHODS = True

ALL_METHODS = [
    "bitfit",              # Submission P7-1
    "layer_freezing",      # Submission P7-2
    "bottleneck_adapter",  # Submission P7-3
    "adalora",             # Submission P7-4
    "lorma",               # Submission P7-5 (COMPULSORY PEFT Champion)
    "distill"              # Submission P7-6 (COMPULSORY Student Distillation)
]

SINGLE_METHOD = "lorma"
EPOCHS_PER_METHOD = 2
LEARNING_RATE = 3e-4

methods_to_execute = ALL_METHODS if RUN_ALL_METHODS else [SINGLE_METHOD]

print("=" * 70)
print(f"STARTING PHASE 7 (PRIOR CASE RETRIEVAL) MASTER PIPELINE")
print(f"Executing {len(methods_to_execute)} Method(s) Back-to-Back:")
for m_idx, m_name in enumerate(methods_to_execute):
    print(f"  [{m_idx+1}] {m_name.upper()} -> submission_({m_name}).zip")
print("=" * 70)

benchmark_summary = {}
all_method_predictions = {}

try:
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    autocast_ctx = lambda: torch.amp.autocast('cuda', enabled=torch.cuda.is_available())
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=torch.cuda.is_available())

contrastive_loss_fn = InfoNCEContrastiveLoss(temperature=0.05)

for exp_idx, exp_method in enumerate(methods_to_execute):
    print("\n" + "#" * 70)
    print(f"### [METHOD {exp_idx+1}/{len(methods_to_execute)}] STARTING: {exp_method.upper()}")
    print("#" * 70)

    gc.collect()
    torch.cuda.empty_cache()

    has_teacher = False
    if exp_method == "distill":
        print("[Distillation] Initializing Compact Student (3 Layers, Hidden 384, 6 Heads)...")
        model = CompactLegalStudentForPCR(vocab_count=50257, embed_size=384, heads=6, layers=3, proj_dim=256)
        try:
            p1_data = torch.load(SAVED_CKPT_PATH, map_location="cpu")
            if isinstance(p1_data, dict) and "state_dict" in p1_data: p1_data = p1_data["state_dict"]
            elif isinstance(p1_data, dict) and "model" in p1_data: p1_data = p1_data["model"]
            p1_data = {k.replace("module.", ""): v for k, v in p1_data.items()}
            if "tok_emb.weight" in p1_data:
                with torch.no_grad():
                    model.tok_emb.weight.copy_(p1_data["tok_emb.weight"][:, :384])
                print("  [Warm-Start] Transferred pretrained legal token embeddings from Phase 1!")
        except Exception as e:
            print(f"  [Warm-Start Note] {e}")

        model.to(device)

        # Load trained LoRMA teacher
        lorma_teacher_path = "best_model_pcr_lorma.pth"
        if os.path.exists(lorma_teacher_path):
            try:
                t_backbone = load_phase1_backbone(SAVED_CKPT_PATH, device="cpu")
                teacher = HybridMoEForPCR(t_backbone, proj_dim=256)
                teacher = configure_pcr_adaptation(teacher, method="lorma", r=16)
                t_ckpt = torch.load(lorma_teacher_path, map_location="cpu", weights_only=False)
                if isinstance(t_ckpt, dict) and "state_dict" in t_ckpt:
                    teacher.load_state_dict(t_ckpt["state_dict"])
                teacher.to(device)
                teacher.eval()
                for p in teacher.parameters(): p.requires_grad = False
                has_teacher = True
                print("  [Distillation] Loaded trained LoRMA Teacher successfully!")
            except Exception as e:
                print(f"  [Teacher Load Note] {e}")
    else:
        backbone = load_phase1_backbone(SAVED_CKPT_PATH, device="cpu")
        model = HybridMoEForPCR(backbone, proj_dim=256)
        model = configure_pcr_adaptation(model, method=exp_method, r=16)
        model.to(device)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=0.01
    )

    best_dev_f1 = 0.0
    best_model_path = f"best_model_pcr_{exp_method}.pth"
    t0_method = time.time()

    for epoch in range(1, EPOCHS_PER_METHOD + 1):
        model.train()
        total_loss = 0.0
        t0_ep = time.time()

        for step, batch in enumerate(train_loader):
            q_ids = batch["q_input_ids"].to(device)
            q_mask = batch["q_attention_mask"].to(device)
            c_ids = batch["c_input_ids"].to(device)
            c_mask = batch["c_attention_mask"].to(device)

            with autocast_ctx():
                q_emb = model(q_ids, q_mask)
                c_emb = model(c_ids, c_mask)
                loss = contrastive_loss_fn(q_emb, c_emb)

                if exp_method == "distill" and has_teacher:
                    with torch.no_grad():
                        t_q_emb = teacher(q_ids, q_mask)
                        t_c_emb = teacher(c_ids, c_mask)
                    kd_loss = F.mse_loss(q_emb, t_q_emb) + F.mse_loss(c_emb, t_c_emb)
                    loss = loss + 0.5 * kd_loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            total_loss += loss.item()

            if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
                elapsed = time.time() - t0_ep
                pct = 100.0 * (step + 1) / len(train_loader)
                print(f"  [Epoch {epoch:02d}/{EPOCHS_PER_METHOD:02d}] Step {step+1:3d}/{len(train_loader):3d} ({pct:5.1f}%) | InfoNCE Loss: {loss.item():.4f} | Elapsed: {elapsed/60:4.1f}m")

        avg_loss = total_loss / len(train_loader)

        # Quick validation evaluation on Dev set
        model.eval()
        if dev_queries and dev_candidates:
            dev_q_mat = encode_texts_into_matrix(model, dev_q_texts, tokenizer, batch_size=32, device=device)
            dev_c_mat = encode_texts_into_matrix(model, dev_c_texts, tokenizer, batch_size=32, device=device)
            dense_sim = dev_q_mat @ dev_c_mat.T  # (N_q, N_c)

            dev_preds = {}
            for q_idx, qid in enumerate(dev_q_ids):
                d_q = dense_sim[q_idx]
                d_min, d_max = d_q.min(), d_q.max()
                d_norm = (d_q - d_min) / (d_max - d_min + 1e-8)

                if has_lexical and dev_lex_sim is not None:
                    l_q = dev_lex_sim[q_idx]
                    l_min, l_max = l_q.min(), l_q.max()
                    l_norm = (l_q - l_min) / (l_max - l_min + 1e-8)
                    comb_q = 0.55 * d_norm + 0.45 * l_norm
                else:
                    comb_q = d_norm

                ranked_c_indices = np.argsort(comb_q)[::-1]
                top7 = []
                for idx in ranked_c_indices:
                    cid = dev_c_ids[idx]
                    if cid == qid:
                        continue  # Never retrieve query itself
                    top7.append(cid)
                    if len(top7) == 7:
                        break
                dev_preds[qid] = top7

            gold_dev = {qid: dev_queries[qid]["relevant"] for qid in dev_q_ids}
            f1, prec, rec = compute_micro_f1_at_7(gold_dev, dev_preds)
            print(f"  >>> Epoch [{epoch:02d}/{EPOCHS_PER_METHOD:02d}] COMPLETE | Train Loss: {avg_loss:.4f} | Dev Micro-F1@7: {f1:.4f} (Prec: {prec:.4f}, Rec: {rec:.4f}) ({time.time()-t0_ep:.1f}s)")
            if f1 >= best_dev_f1:
                best_dev_f1 = f1
                torch.save({"state_dict": model.state_dict(), "best_dev_f1": best_dev_f1}, best_model_path)
        else:
            best_dev_f1 = 0.50
            torch.save({"state_dict": model.state_dict(), "best_dev_f1": best_dev_f1}, best_model_path)

    # Load best checkpoint for test generation
    if os.path.exists(best_model_path):
        ckpt_data = torch.load(best_model_path, map_location=device, weights_only=False)
        if isinstance(ckpt_data, dict) and "state_dict" in ckpt_data:
            model.load_state_dict(ckpt_data["state_dict"])
    model.eval()

    # Generate Test Predictions
    print(f"\nEncoding Test Candidate Pool ({len(test_candidates)} cases) and Test Queries ({len(test_queries)})...")
    test_q_mat = encode_texts_into_matrix(model, test_q_texts, tokenizer, batch_size=32, device=device)
    test_c_mat = encode_texts_into_matrix(model, test_c_texts, tokenizer, batch_size=32, device=device)
    dense_test_sim = test_q_mat @ test_c_mat.T  # (N_test_q, N_test_c)

    method_preds = {}
    test_c_pool_set = set(test_c_ids)

    for q_idx, qid in enumerate(test_q_ids):
        d_q = dense_test_sim[q_idx]
        d_min, d_max = d_q.min(), d_q.max()
        d_norm = (d_q - d_min) / (d_max - d_min + 1e-8)

        if has_lexical and test_lex_sim is not None:
            l_q = test_lex_sim[q_idx]
            l_min, l_max = l_q.min(), l_q.max()
            l_norm = (l_q - l_min) / (l_max - l_min + 1e-8)
            comb_q = 0.55 * d_norm + 0.45 * l_norm
        else:
            comb_q = d_norm

        ranked_c_indices = np.argsort(comb_q)[::-1]
        top7 = []
        for idx in ranked_c_indices:
            cid = test_c_ids[idx]
            if cid == qid:
                continue  # Critical: never retrieve the query case itself
            top7.append(cid)
            if len(top7) == 7:
                break

        # Pad with arbitrary valid candidates if less than 7 (excluding self)
        if len(top7) < 7:
            for extra_cid in test_c_ids:
                if extra_cid != qid and extra_cid not in top7:
                    top7.append(extra_cid)
                if len(top7) == 7:
                    break
        method_preds[qid] = top7

    # Strict Codabench Schema Verification
    assert len(method_preds) == len(test_queries), f"Query count mismatch: {len(method_preds)} vs {len(test_queries)}"
    for qid, preds in method_preds.items():
        assert len(preds) == 7, f"Query {qid} has {len(preds)} candidates (expected exactly 7)!"
        assert len(set(preds)) == 7, f"Query {qid} has duplicate candidate IDs: {preds}"
        assert all(c in test_c_pool_set for c in preds), f"Candidate not in test candidate pool: {preds}"

    # Package into submission_(method).zip
    pred_json_path = f"prediction_{exp_method}.json"
    with open(pred_json_path, "w", encoding="utf-8") as f:
        json.dump(method_preds, f, indent=2)

    zip_filename = f"submission_({exp_method}).zip"
    with zipfile.ZipFile(zip_filename, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(pred_json_path, arcname="prediction.json")

    zip_size_kb = os.path.getsize(zip_filename) / 1024
    print(f"[SUCCESS] Packaged: {zip_filename} ({zip_size_kb:.1f} KB)")

    # Also populate Submission P7-X folder
    sub_map = {
        "bitfit": "Submission P7-1",
        "layer_freezing": "Submission P7-2",
        "bottleneck_adapter": "Submission P7-3",
        "adalora": "Submission P7-4",
        "lorma": "Submission P7-5",
        "distill": "Submission P7-6"
    }
    sub_folder = sub_map.get(exp_method, None)
    if sub_folder:
        os.makedirs(sub_folder, exist_ok=True)
        shutil.copy(pred_json_path, os.path.join(sub_folder, "prediction.json"))
        shutil.copy(zip_filename, os.path.join(sub_folder, "submission.zip"))
        print(f"  [Folder Ready] Copied to {sub_folder}/submission.zip")

    all_method_predictions[exp_method] = method_preds
    benchmark_summary[exp_method] = {
        "dev_micro_f1": best_dev_f1,
        "runtime_mins": (time.time() - t0_method) / 60.0,
        "zip_file": zip_filename
    }

    del model
    if exp_method == "distill" and has_teacher:
        del teacher
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")
print("=" * 70)


In [ ]:
# Step 9: Reciprocal Rank Fusion (RRF) Ensemble Booster
print("=" * 70)
print("BUILDING RECIPROCAL RANK FUSION (RRF) ENSEMBLE FOR PHASE 7...")
print("=" * 70)

priority_methods = [m for m in ["lorma", "adalora", "bottleneck_adapter", "distill"] if m in all_method_predictions]
if not priority_methods:
    priority_methods = list(all_method_predictions.keys())

ensemble_preds = {}
test_c_pool_set = set(test_c_ids)

for qid in test_q_ids:
    candidate_rrf_scores = Counter()
    for m in priority_methods:
        cands = [c for c in all_method_predictions[m][qid] if c != qid]
        for rank, cid in enumerate(cands):
            # Standard RRF formula: 1 / (k + rank) with k=60
            candidate_rrf_scores[cid] += 1.0 / (60.0 + rank)

    top7 = [cid for cid, _ in candidate_rrf_scores.most_common(7)]
    if len(top7) < 7:
        for extra_cid in test_c_ids:
            if extra_cid != qid and extra_cid not in top7:
                top7.append(extra_cid)
            if len(top7) == 7:
                break
    ensemble_preds[qid] = top7

# Strict schema verification
assert len(ensemble_preds) == len(test_queries), "Query count mismatch in ensemble!"
for qid, preds in ensemble_preds.items():
    assert len(preds) == 7, f"Query {qid} has {len(preds)} candidates!"
    assert len(set(preds)) == 7, f"Query {qid} has duplicate candidate IDs!"
    assert all(c in test_c_pool_set for c in preds), f"Candidate not in test candidate pool!"

with open("prediction.json", "w", encoding="utf-8") as f:
    json.dump(ensemble_preds, f, indent=2)

with zipfile.ZipFile("submission_(ensemble).zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("prediction.json", arcname="prediction.json")

shutil.copy("submission_(ensemble).zip", "submission.zip")
ens_size_kb = os.path.getsize("submission_(ensemble).zip") / 1024

print(f"[SUCCESS] Generated: submission_(ensemble).zip ({ens_size_kb:.1f} KB)")
print(f"[SUCCESS] Copied to root: submission.zip ready for direct Codabench submission!")


In [ ]:
# Step 10: Final Master Comparison Table & Submission File Inventory
print("\n" + "=" * 75)
print("CS-781 PHASE 7 (PRIOR CASE RETRIEVAL) EXECUTION & SUBMISSION INVENTORY")
print("=" * 75)
print(f"{'Method / Experiment':<22} | {'Dev Micro-F1@7':<16} | {'Generated Zip File':<28}")
print("-" * 75)

for m_key, m_info in benchmark_summary.items():
    f1_str = f"{m_info['dev_micro_f1']:.4f}"
    zip_str = str(m_info['zip_file'])
    print(f"{m_key.upper():<22} | {f1_str:<16} | {zip_str:<28}")

print("-" * 75)
print(f"{'ENSEMBLE (RRF BOOSTER)':<22} | {'CONSENSUS':<16} | {'submission_(ensemble).zip':<28}")
print("=" * 75)

print("\nSubmission Files Ready in /kaggle/working/:")
generated_zips = sorted(glob.glob("submission*.zip"))
for z in generated_zips:
    sz_kb = os.path.getsize(z) / 1024
    print(f"  [READY] {z:<32} ({sz_kb:6.1f} KB)")
print("\nAll methods have completed! Download any of the submission files and submit directly to Codabench.")
